# 🔎 Data Preview & Signal Structure Analysis

This report presents an Exploratory Data Analysis (EDA) of the EEG Eye State dataset, composed of 14 EEG channel measurements and a binary target indicating eye state (0 = open, 1 = closed). The goal is to assess data structure and quality, understand target distribution, and inspect feature behavior (distributions, outliers, correlations) to inform subsequent preprocessing and modeling decisions.


Examining the first few rows (`head`) reveals three critical technical insights that will guide our preprocessing strategy:

1.  **DC Offset (Baseline Bias):**
    The raw EEG values do not oscillate around zero (as expected in pure electrical signals) but are shifted around a baseline of **~4000-4600**.
    *   *Implication:* This indicates a hardware DC offset. For distance-based algorithms (like kNN or SVM) and Neural Networks, it is mandatory to **center** or **standardize** the data (e.g., `StandardScaler`), otherwise, the model will learn the bias rather than the signal variations.

2.  **Short-Term Stability (Autocorrelation):**
    Values change minimally between consecutive rows (e.g., AF3 moves from 4329 to 4324).
    *   *Implication:* This confirms the high sampling rate (~128Hz). Adjacent samples are strongly **autocorrelated**, meaning we cannot treat them as independent and identically distributed (i.i.d.) observations. This reinforces the need for **time-series validation** (avoiding random shuffle split) to prevent data leakage.

3.  **Target Variable:**
    The `eyeDetection` column is a binary integer (0/1), confirming this is a standard **binary classification task**.



What do the feature columns represent?
Each feature (column) corresponds to the electrical activity of the brain measured at a specific EEG electrode, following the international 10–20 system.

**EEG Feature Columns**

| Column | Brain region | What it represents |
| :--- | :--- | :--- |
| AF3 | Anterior frontal (left) | Attention, focus |
| AF4 | Anterior frontal (right) | Attention, decision making |
| F3 | Frontal (left) | Thinking, memory |
| F4 | Frontal (right) | Cognitive control |
| F7 | Frontal lateral (left) | Emotion processing |
| F8 | Frontal lateral (right) | Emotion processing |
| FC5 | Fronto-central (left) | Motor planning |
| FC6 | Fronto-central (right) | Motor planning |
| T7 | Temporal (left) | Auditory processing |
| T8 | Temporal (right) | Auditory processing |
| P7 | Parietal (left) | Spatial awareness |
| P8 | Parietal (right) | Spatial awareness |
| O1 | Occipital (left) | Visual processing |
| O2 | Occipital (right) | Visual processing |


**✅ Data Integrity Check: No Missing Values**

**Key Finding:** The dataset shows **0% missing values** across all 15 features (14 EEG channels + target).

**Why this matters:**
*   **Positive sign:** This suggests a **controlled laboratory recording** with continuous electrode contact throughout the 117-second session. In real-world EEG data, missing values are common due to:
    *   Electrode detachment
    *   Artifacts leading to signal rejection
    *   Pre-processing filters removing corrupted segments
    
*   **Implication for preprocessing:**
    *   We can skip imputation strategies (no need for mean/median filling or advanced techniques like KNN imputation).
    *   However, we must remain vigilant about **extreme outliers** (which we already noticed in the `describe()` output). Zero missing values doesn't guarantee zero noise.

**Data types:** All features are correctly stored as `float64`, which is suitable for numerical modeling. The target (`eyeDetection`) could be converted to `int8` or `category` to save memory, but for a dataset of ~15K rows, this optimization is negligible.




In [ ]:
# Install required package for accessing UCI datasets - MIO
!pip install ucimlrepo

# Import necessary libraries
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import norm, skew, kurtosis, shapiro
# Fetch the EEG Eye State dataset (ID: 264)
eeg_eye_state = fetch_ucirepo(id=264)


# Extract features and targets as pandas DataFrames
X = eeg_eye_state.data.features  # EEG features (14 columns)
y = eeg_eye_state.data.targets   # Eye state labels (0=open, 1=closed)

# Preview the data
print("\nFirst 5 rows of features (EEG data):")
print(X.head())
print("\nFirst 5 rows of target (eye state):")
print(y.head())

# ============================================================
# DC OFFSET REMOVAL (MEAN CENTERING) + OUTLIER REMOVAL
# ============================================================

print("\n" + "=" * 80)
print("PREPROCESSING: DC OFFSET REMOVAL AND OUTLIER HANDLING")
print("=" * 80)

# Copy original data
X_raw = X.copy()
y_raw = y  # y is already fine (Series or array)

# -------------------------
# STEP 1 – DC OFFSET REMOVAL
# -------------------------

print("\nSTEP 1 – DC OFFSET REMOVAL (Mean Centering)")
print("We subtract the mean of each EEG channel to remove the hardware DC offset.")

# Compute channel-wise means (DC offsets)
dc_offsets = X_raw.mean(axis=0)

print("\nChannel means BEFORE centering (DC offsets, first 5):")
print(dc_offsets.head())

# Mean centering
X_centered = X_raw - dc_offsets

print("\nValue range BEFORE centering: "
      f"[{X_raw.min().min():.2f}, {X_raw.max().max():.2f}] μV")
print("Value range AFTER centering:  "
      f"[{X_centered.min().min():.2f}, {X_centered.max().max():.2f}] μV")

print("\nChannel means AFTER centering (should be ~0, first 5):")
print(X_centered.mean(axis=0).head())

# -------------------------
# STEP 2 – OUTLIER DETECTION & REMOVAL
# -------------------------

print("\n" + "-" * 80)
print("STEP 2 – OUTLIER DETECTION AND REMOVAL")
print("-" * 80)

# Threshold for extreme EEG values (μV)
threshold = 500

print(f"\nOutlier threshold: ±{threshold} μV")

# Boolean mask: rows containing at least one extreme value
mask_with_outliers = (X_centered.abs() > threshold).any(axis=1)
rows_with_outliers = mask_with_outliers.sum()
total_rows = len(X_centered)

print(f"\nRows with outliers (>|{threshold}| μV): "
      f"{rows_with_outliers} / {total_rows} "
      f"({rows_with_outliers/total_rows*100:.3f}%)")

# Keep only rows WITHOUT extreme outliers
mask_clean = ~mask_with_outliers
X_clean = X_centered[mask_clean].copy()
y_clean = y_raw[mask_clean]

print(f"Rows AFTER outlier removal: {len(X_clean)} "
      f"({len(X_clean)/total_rows*100:.2f}% retained)")

# ============================================================
# FROM HERE ON, WE USE THE CLEAN DATA
# ============================================================


# Create a complete DataFrame using CLEANED data
df = X_clean.copy()
df['eye_state'] = y_clean

print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Dataset Shape: {df.shape}")
print(f"Number of EEG Channels: {len(df.columns) - 1}")
print(f"Total Observations: {len(df)}")
print(f"Chronological Recording Duration: 117 seconds")
print(f"Approximate Sampling Rate: {len(df)/117:.1f} Hz")


#basic data profiling
print("\n" + "=" * 80)
print("1. BASIC DATA PROFILING")
print("=" * 80)

# Check data types and missing values
print("\nData Types and Missing Values:")
info_df = pd.DataFrame({
    'Data Type': df.dtypes,
    'Missing Values': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(info_df)

#basic statistic
print("\nDescriptive Statistics:")
print(df.describe().T)

#define eeg_channels
eeg_channels = X_clean.columns.tolist()


print("\n" + "=" * 80)
print("2. CORRELATION ANALYSIS BETWEEN EEG CHANNELS")
print("=" * 80)

# Calculate correlation matrix
corr_matrix = df[eeg_channels].corr()

# Plot correlation heatmap
plt.figure(figsize=(14, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix of EEG Channels (Pearson)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Find strong correlations
strong_corr_threshold = 0.7
strong_correlations = []

for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        corr_value = corr_matrix.iloc[i, j]
        if abs(corr_value) > strong_corr_threshold:
            strong_correlations.append({
                'Channel1': corr_matrix.columns[i],
                'Channel2': corr_matrix.columns[j],
                'Correlation': round(corr_value, 3)
            })

strong_corr_df = pd.DataFrame(strong_correlations)
print(f"\nStrong Correlations (|r| > {strong_corr_threshold}):")
if len(strong_corr_df) > 0:
    print(strong_corr_df.sort_values('Correlation', key=abs, ascending=False).to_string(index=False))
else:
    print("No strong correlations found (|r| < 0.7).")


print("\n" + "=" * 80)
print("3. CORRELATION WITH TARGET VARIABLE")
print("=" * 80)

# CORRELAZIONE CON OCCHI APERTI (target = eye_state == 0)
corr_open = df[eeg_channels].corrwith(df['eye_state'] == 0).sort_values(ascending=False)

plt.figure(figsize=(14, 2))
sns.heatmap(corr_open.values.reshape(1,-1), annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, xticklabels=corr_open.index, yticklabels=['Aperti'],
            vmin=-0.3, vmax=0.3)
plt.title('EEG Correlation with open eyes (1)')
plt.xlabel('EEG Channels')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# CORRELAZIONE CON OCCHI CHIUSI (target = eye_state == 1)
corr_closed = df[eeg_channels].corrwith(df['eye_state'] == 1).sort_values(ascending=False)

plt.figure(figsize=(14, 2))
sns.heatmap(corr_closed.values.reshape(1,-1), annot=True, fmt='.3f', cmap='RdBu_r',
            center=0, xticklabels=corr_closed.index, yticklabels=['Chiusi'],
            vmin=-0.3, vmax=0.3)
plt.title('EEG Correlation with closed eyes (1)')
plt.xlabel('EEG Channels')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# -------------------------
# STEP 3 – VISUAL INSPECTION (BOXPLOTS BEFORE/AFTER)
# -------------------------
# [boxplot function]
# Helper function to plot boxplots in a 4x4 grid
def plot_boxplots(data, title):
    
    fig, axes = plt.subplots(4, 4, figsize=(16, 14))
    axes = axes.ravel()
    
    for i, channel in enumerate(eeg_channels):
        ax = axes[i]
        
        bp = ax.boxplot(data[channel], vert=True, patch_artist=True)
        
        # Style
        plt.setp(bp['fliers'], marker='o', color='red', alpha=0.5, markersize=3)
        plt.setp(bp['boxes'], facecolor='lightblue')
        
        ax.set_title(channel, fontsize=10, fontweight='bold')
        ax.set_ylabel('Amplitude (μV)')
        ax.grid(True, alpha=0.3)
    
    
    for j in range(len(eeg_channels), 16):
        axes[j].set_visible(False)
    
    
    plt.suptitle(title, fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

plot_boxplots(X_centered, "BEFORE Outlier Removal")
plot_boxplots(X_clean, "AFTER Outlier Removal")

# -------------------------
# STEP 4 – DESCRIPTIVE OUTLIER ANALYSIS (IQR & Z-SCORE)
# -------------------------
print("\n" + "-" * 80)
print("DESCRIPTIVE ANALYSIS OF RESIDUAL OUTLIERS (IQR vs Z-SCORE)")
print("-" * 80)

# Function to detect outliers using IQR method
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Function to detect outliers using Z-score method
def detect_outliers_zscore(data, column, threshold=3):
    z_scores = np.abs(stats.zscore(data[column]))
    outliers = data[z_scores > threshold]
    return outliers

outlier_summary = pd.DataFrame(columns=['Channel', 'IQR_count', 'IQR_%', 'Z_count', 'Z_%', 'IQR_bounds'])

for channel in eeg_channels[:8]:
    iqr_outliers, lower_b, upper_b = detect_outliers_iqr(X_clean, channel)
    z_outliers = detect_outliers_zscore(X_clean, channel)
    
    outlier_summary = pd.concat([outlier_summary, pd.DataFrame([{
        'Channel': channel,
        'IQR_count': len(iqr_outliers),
        'IQR_%': f"{len(iqr_outliers)/len(X_clean)*100:.2f}%",
        'Z_count': len(z_outliers),
        'Z_%': f"{len(z_outliers)/len(X_clean)*100:.2f}%",
        'IQR_bounds': f'[{lower_b:.0f}, {upper_b:.0f}]'
    }])], ignore_index=True)

print(outlier_summary.to_string(index=False))


# EEG channels distribution


In [ ]:
# ============================================================
# TARGET & DISTRIBUTION ANALYSIS (UNIFIED)
# ============================================================

print("\n" + "=" * 80)
print("1. TARGET VARIABLE ANALYSIS")
print("=" * 80)

# Define distribution
eye_state_counts = df['eye_state'].value_counts().sort_index()
eye_state_percent = df['eye_state'].value_counts(normalize=True) * 100
imbalance_ratio = eye_state_counts[0] / eye_state_counts[1]

print(f"Eye Open (0): {eye_state_counts[0]} samples ({eye_state_percent[0]:.2f}%)")
print(f"Eye Closed (1): {eye_state_counts[1]} samples ({eye_state_percent[1]:.2f}%)")
print(f"Class Imbalance Ratio (Open/Closed): {imbalance_ratio:.3f}")

# Layout
fig = plt.figure(figsize=(16, 20))
gs = fig.add_gridspec(6, 4, hspace=0.4, wspace=0.3)

# --- Subplot for the Target 
ax_target = fig.add_subplot(gs[0, 1:3])
colors_target = ['lightblue', 'lightcoral']
bars = ax_target.bar(['Open (0)', 'Closed (1)'], eye_state_counts.values, color=colors_target, edgecolor='black', alpha=0.8)
ax_target.set_title('Eye State Distribution', fontsize=12, fontweight='bold')
ax_target.set_ylabel('Count')

# label
for bar in bars:
    height = bar.get_height()
    ax_target.text(bar.get_x() + bar.get_width()/2., height + 100, f'{int(height)}', ha='center', va='bottom', fontweight='bold')

# --- Subplots for EEG Channels
eeg_channels = [col for col in df.columns if col != 'eye_state']
colors_eeg = sns.color_palette("viridis", len(eeg_channels))

# Start from row 1
for i, channel in enumerate(eeg_channels):
    row = (i // 4) + 1
    col = i % 4
    ax = fig.add_subplot(gs[row, col])
    
    sns.histplot(x=df[channel], bins=50, kde=True, color=colors_eeg[i], ax=ax, alpha=0.6)
    
    ax.set_title(f"Channel {channel}", fontsize=10, fontweight='bold')
    ax.set_xlabel("Amplitude (μV)")
    ax.set_ylabel("Freq")
    ax.grid(True, alpha=0.2)

plt.suptitle('EEG Eye State: Target and Feature Distributions (Cleaned Data)', 
             fontsize=18, fontweight='bold', y=0.95)


for j in range(len(eeg_channels), 20):
    row = (j // 4) + 1
    if row < 6:
        col = j % 4


plt.show()

# MEAN EEG AMPLITUDE PER CHANNEL BY EYE STATE

In [ ]:
# 1

df_abs = df.copy()
for ch in eeg_channels:
    df_abs[ch] = df_abs[ch].abs()

means_abs = df_abs.groupby('eye_state')[eeg_channels].mean()
stds_abs = df_abs.groupby('eye_state')[eeg_channels].std()

eye_open_means = means_abs.loc[0]
eye_closed_means = means_abs.loc[1]
eye_open_stds = stds_abs.loc[0]
eye_closed_stds = stds_abs.loc[1]

# 2. 
fig, ax = plt.subplots(figsize=(14, 7))

x = np.arange(len(eeg_channels))
width = 0.35

# 
ax.bar(x - width/2, eye_open_means, width, label='Eye Open (0)', 
       yerr=eye_open_stds, capsize=5, alpha=0.7, color='skyblue', edgecolor='black')
ax.bar(x + width/2, eye_closed_means, width, label='Eye Closed (1)', 
       yerr=eye_closed_stds, capsize=5, alpha=0.7, color='salmon', edgecolor='black')

ax.set_ylabel('Mean Absolute Amplitude (μV)', fontsize=12, fontweight='bold')
ax.set_title('EEG Amplitude Analysis: Mean ± STD per State', fontsize=15, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(eeg_channels, rotation=45)
ax.axhline(0, color='black', linewidth=0.8) 
ax.legend()
ax.grid(True, alpha=0.2, axis='y')

plt.tight_layout()
plt.show()
